# Step 3 — Prior team's baseline (Hybrid NCF + LSTM) and gap diagnosis

Re-implements the previous team's model, evaluates it on the same full slates as every other model, and measures the structural reasons it underperforms.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/harvestingmoon/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 3.1 The model (`scripts/02c_prior_ncf.py`, `src/jds/prior_ncf.py`)

Their setup: per-candidate 4-class classification trained on lists subsampled to 1 positive : 4 negatives; user = 2-layer
LSTM + attention over the last 200 history events plus 8 user statistics (click/cart/order ratios over the whole
history); candidate = item id + brand/shop/category embeddings. Two variants:
* `published` — as released: the attribute lookup indexes the product table with the *history* item index (a bug),
  ranked by the max class probability;
* `fixed` — the item's own attributes, ranked by the expected label.

Both are scored on the full displayed slates of test users, and on their own sampled 1:4 lists for reference (which is
why their originally reported numbers look much better than on the real task).

In [ ]:
run.stage("02c_prior_ncf.py")

In [ ]:
banner()
import pandas as pd
pd.DataFrame(run.json("prior_ncf.json")).style.format(precision=4).highlight_max(subset=["mrr", "p1", "ndcg10", "gndcg10", "hr10"], color="#c6efce")

## 3.2 Gap diagnosis

The measured structural gaps (`results/prior_ncf_gaps.json`): the test query is never an input; share of displayed items
(and of positives) with no item embedding because they never occur in history; how often the `published` lookup gives an
item someone else's attributes; training lists of ~5 items vs ~89-item test slates; history truncated to 200 events;
user ratios that mix retention windows (see step 1, recency confound).

In [ ]:
run.json("prior_ncf_gaps.json")